# Cloud GPU environment check (Colab)

Quick check that the CUDA side of the stack works before I rely on Colab
for the bigger jobs (large index builds, heavier generation). Locally I
generate with MLX 4-bit on the Mac; on Colab the same 7B models load
through transformers with bitsandbytes 4-bit. Set the runtime to GPU
before running anything (Runtime, Change runtime type, T4 is enough).


In [1]:
!nvidia-smi

Sun Aug 16 03:05:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Colab already ships torch+CUDA. Add the system libs (skip torch reinstall).
!pip -q install transformers accelerate bitsandbytes sentence-transformers \
  faiss-cpu rank-bm25 'langgraph>=0.2' 'langchain>=0.2' pydantic pydantic-settings datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 21.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 102.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.5 MB/s eta 0:00:00


## Stack check

Imports, a real matmul on the GPU, a FAISS search and a minimal LangGraph
run. If any of these fail there is no point going further.


In [3]:
import torch, platform
print('python', platform.python_version(), '| machine', platform.machine())
assert torch.cuda.is_available(), 'no CUDA GPU — set Runtime to GPU'
x = torch.randn(1024,1024,device='cuda'); print('cuda matmul ok:', float((x@x).sum()) == float((x@x).sum()))
print('gpu:', torch.cuda.get_device_name(0))

python 3.12.13 | machine x86_64
cuda matmul ok: True
gpu: Tesla T4


In [4]:
import transformers, accelerate, sentence_transformers, faiss, numpy as np
from rank_bm25 import BM25Okapi
print('transformers', transformers.__version__, '| accelerate', accelerate.__version__)
idx=faiss.IndexFlatIP(16); idx.add(np.random.rand(8,16).astype('float32'))
print('faiss search ok:', idx.search(np.random.rand(1,16).astype('float32'),3)[1].shape)

transformers 5.13.1 | accelerate 1.14.0
faiss search ok: (1, 3)


In [5]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
class S(TypedDict):
    n:int
g=StateGraph(S); g.add_node('inc', lambda s:{'n':s['n']+1}); g.set_entry_point('inc'); g.add_edge('inc',END)
print('langgraph invoke:', g.compile().invoke({'n':0}))

langgraph invoke: {'n': 1}


## Optional: load a 7B generator in 4-bit

This is the CUDA replacement for the local MLX path. Downloads about 5 GB,
so skip it when the session is only a quick check.


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
mid='Qwen/Qwen2.5-7B-Instruct'
bnb=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
tok=AutoTokenizer.from_pretrained(mid)
model=AutoModelForCausalLM.from_pretrained(mid, quantization_config=bnb, device_map='auto')
ids=tok('Reply with one word: ok', return_tensors='pt').to(model.device)
print(tok.decode(model.generate(**ids, max_new_tokens=8)[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Reply with one word: ok
Ok.


All green means the stack I use locally also runs on CUDA, so Colab can
take the large index builds and the evaluation sweeps while day to day
development stays on the laptop (conda activate adarag).
